First I handle the stitch - writing a simple for loop that reads every .json file in the raw data folder and merges them together into one larger Pandas DataFrame.

Then I'll extract the exact date each article was published.

In [4]:
import pandas as pd
import json
import glob
import os
import sys

# 1. Load the custom cleaning pipeline
sys.path.append(os.path.abspath('..'))
from src.features import process_raw_to_clean

# 2. Finding all JSON files in the raw data folder
# using glob to look for a specific pattern, merging every daily file I saved
file_pattern = '../data/raw/*_technology_news.json'
all_files = glob.glob(file_pattern)

print(f"Found {len(all_files)} days of news data.")

all_articles = []
for file_path in all_files:
    with open(file_path, 'r') as f:
        data = json.load(f)
        # Extending the master list with the articles from this specific day
        all_articles.extend(data.get('articles', []))

# Converting this to a single larger pandas DataFrame
df_raw = pd.DataFrame(all_articles)
print(f"Total raw articles merged: {len(df_raw)}")

# 3. Cleaning the stitched data using the features.py script
df_clean = process_raw_to_clean(df_raw).copy()

# 4. NewsAPI provides a 'publishedAt' timestamp. I need to format it cleanly
# so that BERTopic knows exactly when each article was written.
df_clean['date'] = pd.to_datetime(df_clean['publishedAt']).dt.date

# Prepare the two lists BERTopic needs: the text and the timestamps
docs = df_clean['text_cleaned'].tolist()
timestamps = df_clean['date'].tolist()

print(f"Ready to track {len(docs)} clean articles over time.\n")

Found 7 days of news data.
Total raw articles merged: 364
Ready to track 357 clean articles over time.



Now this is where I do the dynamic modelling (BERTopic)

In [6]:
from dotenv import load_dotenv
from pathlib import Path

# Loading Hugging Face token from .env file
env_path = Path.cwd().parent / ".env"
load_dotenv(dotenv_path=env_path)

from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
from sklearn.feature_extraction.text import CountVectorizer

# 1. Initialise the optimised components
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")
vectorizer_model = CountVectorizer(stop_words="english", ngram_range=(1, 2))

# 2. Training the core model (same as I did before)
topic_model = BERTopic(
    embedding_model=embedding_model,
    vectorizer_model=vectorizer_model,
    min_topic_size=5
)

print("Training base topic model...")
topics, probs = topic_model.fit_transform(docs)

# 3. Calculate Topics Over Time
# This function recalculates the c-TF-IDF score for every topic at every timestamp.
print("Calculating topic drift...")
topics_over_time = topic_model.topics_over_time(docs, timestamps)

# 4. Visualise the drift!
# top_n_topics limits the graph to the 5 most popular themes so it isn't too messy.
fig_drift = topic_model.visualize_topics_over_time(topics_over_time, top_n_topics=5)
fig_drift.show()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7000.13it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Training base topic model...
Calculating topic drift...
